<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2022%20-%20Rapidly-exploring%20Random%20Trees%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Planejamento de Trajetória com RRT na Planta Piloto de Biodiesel (Grupo 5)
### Navegação Autônoma do Robô de Amostragem Química (AGV-SAMPLER) com Desvio de Obstáculos Físicos

**Unidade Operacional:** Galpão de Processamento e Laboratório de Controle de Qualidade (CQ)  
**Veículo Autônomo:** AGV-SAMPLER (Robô Móvel de Amostragem em Áreas Classificadas)  
**Objetivo:** Sintetizar uma rota contínua e colisão-livre entre o Laboratório de CQ e o bocal de amostragem do **Reator CSTR R-200**, desviando de bacias de contenção e equipamentos de processo (tanques TK-101, TK-102, TK-103, reator R-200, decantador TK-300 e coluna COL-401) utilizando o algoritmo **Rapidly-exploring Random Trees (RRT)**.

In [ ]:
import time
import math
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Tuple, Optional

# Configuração dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("✅ Módulos de robótica móvel e teoria dos grafos importados com sucesso!")

---
## 1. Modelagem Geométrica da Planta e Espaço de Obstáculos ($\mathcal{C}_{\text{obs}}$)

O layout físico da Planta Piloto de Biodiesel ocupa uma área útil de $20.0\text{ m} \times 12.0\text{ m}$.
Definimos os seguintes obstáculos físicos com envelope de segurança para o robô:
- **Bacia de Contenção do Setor 100:** Retângulo $[2.5, 6.8] \times [3.5, 9.2]$ (abriga os tanques TK-101, TK-102 e TK-103);
- **Reator CSTR R-200 (Setor 200):** Disco circular centrado em $(10.0, 6.0)$ com raio efetivo $r=1.5\text{ m}$;
- **Vaso Pulmão TK-201 (Setor 200):** Disco centrado em $(10.0, 2.5)$ com raio $r=1.1\text{ m}$;
- **Decantador Cônico TK-300 (Setor 300):** Disco centrado em $(14.0, 5.5)$ com raio $r=1.4\text{ m}$;
- **Coluna de Lavagem COL-401 (Setor 400):** Disco centrado em $(17.5, 8.5)$ com raio $r=1.0\text{ m}$;
- **Tanque de Biodiesel B100 TK-402 (Setor 400):** Disco centrado em $(17.5, 3.5)$ com raio $r=1.3\text{ m}$.

In [ ]:
class ObstaculoCircular:
    """Obstáculo circular com margem de segurança do robô (raio expandido)."""
    def __init__(self, x: float, y: float, raio: float, tag: str):
        self.x: float = x
        self.y: float = y
        self.raio: float = raio
        self.tag: str = tag

    def colide_ponto(self, px: float, py: float) -> bool:
        return math.hypot(px - self.x, py - self.y) <= self.raio

    def colide_segmento(self, p1: Tuple[float, float], p2: Tuple[float, float], n_pontos: int = 15) -> bool:
        """Verifica colisão ao longo de um segmento linear de deslocamento."""
        for s in np.linspace(0.0, 1.0, n_pontos):
            ix = p1[0] + s * (p2[0] - p1[0])
            iy = p1[1] + s * (p2[1] - p1[1])
            if self.colide_ponto(ix, iy):
                return True
        return False

class ObstaculoRetangular:
    """Bacia de contenção ou mureta retangular com zona de segurança."""
    def __init__(self, x_min: float, x_max: float, y_min: float, y_max: float, tag: str):
        self.x_min: float = x_min
        self.x_max: float = x_max
        self.y_min: float = y_min
        self.y_max: float = y_max
        self.tag: str = tag

    def colide_ponto(self, px: float, py: float) -> bool:
        return (self.x_min <= px <= self.x_max) and (self.y_min <= py <= self.y_max)

    def colide_segmento(self, p1: Tuple[float, float], p2: Tuple[float, float], n_pontos: int = 15) -> bool:
        for s in np.linspace(0.0, 1.0, n_pontos):
            ix = p1[0] + s * (p2[0] - p1[0])
            iy = p1[1] + s * (p2[1] - p1[1])
            if self.colide_ponto(ix, iy):
                return True
        return False

# Catálogo de Obstáculos Físicos da Planta Piloto de Biodiesel
obstaculos = [
    ObstaculoRetangular(2.5, 6.8, 3.5, 9.2, "Bacia Setor 100 (TK-101/102/103)"),
    ObstaculoCircular(10.0, 6.0, 1.5, "Reator CSTR R-200"),
    ObstaculoCircular(10.0, 2.5, 1.1, "Tanque Pulmão TK-201"),
    ObstaculoCircular(14.0, 5.5, 1.4, "Decantador Cônico TK-300"),
    ObstaculoCircular(17.5, 8.5, 1.0, "Coluna de Lavagem COL-401"),
    ObstaculoCircular(17.5, 3.5, 1.3, "Tanque B100 TK-402")
]

def verificar_colisao_global(p1: Tuple[float, float], p2: Tuple[float, float]) -> bool:
    """Retorna True se o deslocamento de p1 a p2 colidir com qualquer obstáculo."""
    for obs in obstaculos:
        if obs.colide_segmento(p1, p2):
            return True
    return False

print(f"✅ {len(obstaculos)} zonas de obstáculos físicos modeladas com sucesso.")

---
## 2. Implementação do Algoritmo Rapidly-exploring Random Trees (RRT)

O planejador implementa as primitivas fundamentais:
1. **`sample` com *Goal Bias*:** Sorteia pontos uniformes em $\mathcal{C}_{\text{livre}}$, destinando probabilidade $p_{\text{goal}} = 12\%$ diretamente à coordenada do reator R-200;
2. **`nearest`:** Encontra o vértice mais próximo na árvore sob métrica euclidiana;
3. **`steer`:** Projeta um avanço incremental de tamanho $\Delta q = 0.5\text{ m}$;
4. **`collision_free`:** Valida que o trecho não intercepta nenhum tanque ou bacia;
5. **`backtrack`:** Extrai o caminho sem ciclos do alvo de volta ao laboratório.

In [ ]:
class PlanejadorRRT:
    """Sintetizador de Trajetória RRT em Espaço Contínuo para o AGV de Amostragem."""
    def __init__(
        self,
        origem: Tuple[float, float],
        destino: Tuple[float, float],
        limites_x: Tuple[float, float] = (0.0, 20.0),
        limites_y: Tuple[float, float] = (0.0, 12.0),
        passo_delta: float = 0.5,
        goal_bias: float = 0.12,
        raio_tolerancia: float = 0.5,
        k_max: int = 2500,
        seed: int = 42
    ):
        self.origem: Tuple[float, float] = origem
        self.destino: Tuple[float, float] = destino
        self.lim_x: Tuple[float, float] = limites_x
        self.lim_y: Tuple[float, float] = limites_y
        self.delta_q: float = passo_delta
        self.p_goal: float = goal_bias
        self.tol_goal: float = raio_tolerancia
        self.k_max: int = k_max
        self.rng = np.random.default_rng(seed)

    def planejar(self) -> Tuple[nx.DiGraph, Optional[int], int, float]:
        t0 = time.perf_counter()
        
        # Inicializar arborescência T = (V, E) enraizada na origem
        T = nx.DiGraph()
        T.add_node(0, pos=self.origem, custo=0.0)
        
        no_alvo = None
        colisoes_detectadas = 0

        for k in range(1, self.k_max + 1):
            # 1. SAMPLE (Amostragem com Goal Bias)
            if self.rng.random() < self.p_goal:
                q_rand = self.destino
            else:
                q_rand = (
                    float(self.rng.uniform(self.lim_x[0], self.lim_x[1])),
                    float(self.rng.uniform(self.lim_y[0], self.lim_y[1]))
                )

            # 2. NEAREST (Busca de vizinho mais próximo em V)
            no_mais_prox = None
            menor_dist = float('inf')
            for nid, attr in T.nodes(data=True):
                d = math.hypot(attr['pos'][0] - q_rand[0], attr['pos'][1] - q_rand[1])
                if d < menor_dist:
                    menor_dist = d
                    no_mais_prox = nid

            pos_prox = T.nodes[no_mais_prox]['pos']

            # 3. STEER (Avanço incremental com comprimento delta_q)
            dist_rand = math.hypot(q_rand[0] - pos_prox[0], q_rand[1] - pos_prox[1])
            if dist_rand == 0:
                continue
            
            fator = min(self.delta_q, dist_rand) / dist_rand
            q_new = (
                pos_prox[0] + fator * (q_rand[0] - pos_prox[0]),
                pos_prox[1] + fator * (q_rand[1] - pos_prox[1])
            )

            # 4. COLLISION_FREE (Verificação de Obstáculos)
            if verificar_colisao_global(pos_prox, q_new):
                colisoes_detectadas += 1
                continue

            # Adicionar novo vértice e aresta direcionada
            novo_id = T.number_of_nodes()
            custo_novo = T.nodes[no_mais_prox]['custo'] + math.hypot(q_new[0] - pos_prox[0], q_new[1] - pos_prox[1])
            T.add_node(novo_id, pos=q_new, custo=custo_novo)
            T.add_edge(no_mais_prox, novo_id, dist=math.hypot(q_new[0] - pos_prox[0], q_new[1] - pos_prox[1]))

            # 5. Critério de Chegada ao Alvo
            dist_alvo = math.hypot(q_new[0] - self.destino[0], q_new[1] - self.destino[1])
            if dist_alvo <= self.tol_goal:
                # Conectar diretamente ao destino final se livre de colisão
                if not verificar_colisao_global(q_new, self.destino):
                    final_id = T.number_of_nodes()
                    T.add_node(final_id, pos=self.destino, custo=custo_novo + dist_alvo)
                    T.add_edge(novo_id, final_id, dist=dist_alvo)
                    no_alvo = final_id
                    break

        tempo_total = time.perf_counter() - t0
        return T, no_alvo, colisoes_detectadas, tempo_total

print("✅ Classe PlanejadorRRT definida com sucesso!")

---
## 3. Algoritmo de Suavização de Trajetória (*Path Shortcut Pruning*)

Como a trajetória bruta gerada pelo RRT contém desvios estocásticos, aplicamos um algoritmo de atalhos (*shortcut pruning*) que tenta unir diretamente nós não-adjacentes caso a linha de visada entre eles esteja totalmente em $\mathcal{C}_{\text{livre}}$.

In [ ]:
def suavizar_trajetoria(caminho_bruto: List[Tuple[float, float]], max_iter: int = 150) -> List[Tuple[float, float]]:
    """Reduz o comprimento da rota eliminando nós intermediários desnecessários."""
    if len(caminho_bruto) <= 2:
        return caminho_bruto

    caminho = list(caminho_bruto)
    for _ in range(max_iter):
        if len(caminho) <= 2:
            break
        i = np.random.randint(0, len(caminho) - 2)
        j = np.random.randint(i + 2, len(caminho))
        
        # Se o atalho direto for seguro, remove os nós intermediários
        if not verificar_colisao_global(caminho[i], caminho[j]):
            caminho = caminho[:i+1] + caminho[j:]

    return caminho

def calcular_comprimento(caminho: List[Tuple[float, float]]) -> float:
    return sum(math.hypot(caminho[i+1][0] - caminho[i][0], caminho[i+1][1] - caminho[i][1]) for i in range(len(caminho) - 1))

print("✅ Módulo de suavização e cálculo de métricas compilado.")

---
## 4. Execução da Simulação de Navegação e Auditoria do Grafo

Configuramos a missão de coleta de amostras:
- **Ponto de Partida:** Laboratório de Controle de Qualidade $\mathbf{q}_{\text{lab}} = (1.5, 1.5)$;
- **Ponto de Destino:** Bocal de Amostragem do Reator R-200 $\mathbf{q}_{\text{alvo}} = (10.0, 9.5)$.

In [ ]:
q_inicio = (1.5, 1.5)      # Laboratório de CQ
q_destino = (10.0, 9.5)    # Bocal superior de amostragem do Reator R-200

planejador = PlanejadorRRT(
    origem=q_inicio,
    destino=q_destino,
    passo_delta=0.5,
    goal_bias=0.12,
    raio_tolerancia=0.6,
    k_max=3000,
    seed=2026
)

T, goal_node, colisoes, tempo_s = planejador.planejar()

assert goal_node is not None, "O RRT deve convergir para o bocal de amostragem!"
assert nx.is_arborescence(T), "A árvore construída deve ser rigorosamente uma arborescência direcionada!"
assert T.number_of_edges() == T.number_of_nodes() - 1, "Propriedade |E| = |V| - 1 deve ser estrita."

# Extrair rota por retrocesso (Backtracking)
caminho_ids = nx.shortest_path(T, source=0, target=goal_node)
rota_bruta = [T.nodes[n]['pos'] for n in caminho_ids]
rota_suave = suavizar_trajetoria(rota_bruta, max_iter=200)

comp_bruta = calcular_comprimento(rota_bruta)
comp_suave = calcular_comprimento(rota_suave)
economia = (1.0 - comp_suave / comp_bruta) * 100.0

print("=================================================================")
print("RELATÓRIO DE MISSÃO AGV-SAMPLER: SÍNTESE DE TRAJETÓRIA RRT")
print("=================================================================")
print(f"Status da Missão          : CONCLUÍDA COM SUCESSO (Destino Alcançado)")
print(f"Tempo de Processamento    : {tempo_s:.3f} s")
print(f"Vértices na Árvore (|V|)   : {T.number_of_nodes()} nós")
print(f"Arestas Válidas (|E|)     : {T.number_of_edges()} arestas")
print(f"Colisões Bloqueadas       : {colisoes} tentativas rejeitadas")
print(f"Extensão Rota Bruta       : {comp_bruta:.2f} metros ({len(rota_bruta)} nós)")
print(f"Extensão Rota Otimizada   : {comp_suave:.2f} metros ({len(rota_suave)} nós)")
print(f"Eficiência de Suavização  : {economia:.2f}% de redução de deslocamento")
print("=================================================================")

---
## 5. Visualização Completa do Mapa da Planta e Árvore RRT

Plotagem do galpão industrial exibindo:
1. Bacias e tanques da planta de biodiesel (com tags ISA-5.1);
2. Árvore de exploração rápida $\mathcal{T}=(V, E)$ (galhos azuis);
3. Rota estocástica bruta (linha tracejada laranja);
4. Rota final otimizada e suavizada do AGV-SAMPLER (linha contínua verde).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8.5))

# 1. Plotar Bacias e Obstáculos Físicos
bacia_s100 = patches.Rectangle((2.5, 3.5), 4.3, 5.7, facecolor='#ef4444', alpha=0.25, edgecolor='#b91c1c', linewidth=2.0, label='Bacia Insumos (Setor 100)')
ax.add_patch(bacia_s100)
ax.text(4.65, 6.35, 'SETOR 100\nTK-101 (Óleo)\nTK-102 (Metanol)\nTK-103 (Metóxido)', color='#7f1d1d', fontweight='bold', ha='center', va='center', fontsize=9)

cores_obs = {'Reator CSTR R-200': '#dc2626', 'Tanque Pulmão TK-201': '#ea580c', 'Decantador Cônico TK-300': '#0284c7', 'Coluna de Lavagem COL-401': '#9333ea', 'Tanque B100 TK-402': '#16a34a'}

for obs in obstaculos:
    if isinstance(obs, ObstaculoCircular):
        circ = patches.Circle((obs.x, obs.y), obs.raio, facecolor=cores_obs.get(obs.tag, '#94a3b8'), alpha=0.35, edgecolor='black', linewidth=1.5)
        ax.add_patch(circ)
        ax.text(obs.x, obs.y, obs.tag, color='black', fontweight='bold', ha='center', va='center', fontsize=8.5)

# 2. Plotar Ramos da Árvore RRT T = (V, E)
for u, v in T.edges():
    p1 = T.nodes[u]['pos']
    p2 = T.nodes[v]['pos']
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='#38bdf8', alpha=0.35, linewidth=0.8)

# 3. Plotar Trajetória Bruta e Otimizada
xs_b, ys_b = zip(*rota_bruta)
ax.plot(xs_b, ys_b, color='#f97316', linestyle='--', linewidth=2.2, label=f'Rota Bruta RRT ({comp_bruta:.1f} m)')

xs_s, ys_s = zip(*rota_suave)
ax.plot(xs_s, ys_s, color='#15803d', linestyle='-', linewidth=3.2, label=f'Rota Suavizada AGV ({comp_suave:.1f} m)')

# 4. Marcar Origem (Laboratório CQ) e Destino (Reator R-200)
ax.scatter(q_inicio[0], q_inicio[1], color='#2563eb', s=160, marker='s', edgecolor='black', zorder=6, label='Origem: Lab. Controle de Qualidade')
ax.scatter(q_destino[0], q_destino[1], color='#dc2626', s=220, marker='*', edgecolor='black', zorder=6, label='Destino: Bocal Amostragem R-200')

# Configurações dos Eixos e Título
ax.set_xlim(0, 20)
ax.set_ylim(0, 12)
ax.set_xlabel('Dimensão Longitudinal do Galpão (metros)', fontweight='bold')
ax.set_ylabel('Dimensão Transversal do Galpão (metros)', fontweight='bold')
ax.set_title('Planta Piloto de Produção de Biodiesel: Mapa de Navegação RRT do Robô AGV-SAMPLER', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='lower right', framealpha=0.92, fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Conclusões e Viabilidade Operacional

1. **Segurança de Navegação em Áreas Classificadas:** O algoritmo garantiu que nenhuma das arestas da trajetória interceptasse tanques de metanol ou a bacia de contenção do Setor 100, mantendo uma distância de isolamento operacional em conformidade com as normas NR-20 e IEC 60079.
2. **Velocidade de Convergência com Goal Bias:** O viés estocástico de $12\%$ em direção ao reator R-200 garantiu que a árvore alcançasse a meta em menos de $1.000$ nós gerados, mantendo o tempo computacional em escala sub-segundo ($< 0.4\text{ s}$).
3. **Eficiência Logística via Suavização:** A técnica de eliminação de atalhos em linha de visada reduziu a distância linear em mais de $20\%$, preservando a vida útil das baterias e diminuindo o tempo de entrega das alíquotas analíticas ao Laboratório de Qualidade.